# M05 — Window functions

[← Anterior](../M04-integracion-agregacion/04-lab-segmentacion.ipynb) · [Siguiente →](02-lab-ranking-ventana.ipynb)

`groupBy` **aplasta** filas. Una window **calcula y conserva** el detalle.

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

Ejecuta estas dos celdas. Localizan el repo y dejan una `SparkSession` lista.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark('novashop-clase-m05')
print(spark.version, spark.sparkContext.master)


## `row_number` y suma acumulada

`partitionBy` de la window ≠ `repartition` físico (eso es M06).


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, row_number, sum as fsum
from pyspark.sql.window import Window

hist = spark.createDataFrame([
    Row(customer_id="C1", order_id="O1", order_n_ts="2024-01-01", gmv=10.0),
    Row(customer_id="C1", order_id="O2", order_n_ts="2024-02-01", gmv=30.0),
    Row(customer_id="C2", order_id="O3", order_n_ts="2024-01-15", gmv=5.0),
    Row(customer_id="C2", order_id="O4", order_n_ts="2024-03-01", gmv=8.0),
])
w = Window.partitionBy("customer_id").orderBy("order_n_ts")
(
    hist.withColumn("order_n", row_number().over(w))
    .withColumn("gmv_running", fsum("gmv").over(w))
    .orderBy("customer_id", "order_n")
    .show()
)


**Siguiente:** [lab de ranking](02-lab-ranking-ventana.ipynb).
